# Lysosomal Trapping PBPK Model
**Intracellular pH Partitioning · Basic Lipophilic Drugs · Extended PBPK | OSP MoBi Exercise**

**Author:** Nadia Tasnim Ahmed, PhD  
**Field:** PBPK Modeling · Intracellular Pharmacokinetics · Drug Distribution  
**Tools:** Python · numpy · scipy · pandas · matplotlib · plotly  
**Reference:** OSP MoBi Course — Lysosomal Trapping (v12)

---

## Background

**Lysosomes** are intracellular organelles with highly acidic pH (4.5–5.0).
Basic lipophilic drugs accumulate massively inside lysosomes through ion trapping:

```
Plasma (pH 7.4)  →  Cytosol (pH 7.2)  →  Lysosome (pH 4.5)
  Drug (neutral)      Drug (neutral)       Drug-H+ (charged, trapped!)
  freely permeable    freely permeable     cannot cross membrane
```

**Henderson-Hasselbalch pH partitioning:**
$$K_{lys/cyt} = 1 + 10^{pKa - pH_{lysosome}} \quad \text{(basic drug)}$$

For chloroquine (pKa=10.1, pH_lys=4.5):
$$K_{lys/cyt} = 1 + 10^{10.1 - 4.5} = 1 + 10^{5.6} \approx 400,000$$

**Clinical significance:**
- Massive tissue accumulation (Vd often >100 L/kg for chloroquine)
- Prolonged tissue half-life despite short plasma half-life
- Lysosomal storage diseases: drug can disrupt lysosomal function
- Drug-drug interactions: lysosomotropic agents compete for lysosomal space
- Drug-induced phospholipidosis (DIPL): lysosomal overload

**In MoBi:**
- Extend PBPK model by adding lysosomal sub-compartment to each tissue
- Partition coefficient Klysosome/cytosol from Henderson-Hasselbalch
- Passive transport between cytosol and lysosome (neutral form only)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.integrate import odeint
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries loaded.')

## 1. Drug Properties & pH Partitioning Theory

In [ ]:
# pH values of compartments
PH = dict(
    plasma    = 7.40,
    cytosol   = 7.20,
    lysosome  = 4.50,  # highly acidic
    endo_lyso = 5.50,  # early endosome
    mitochond = 8.00,  # alkaline (relevant for basic drugs too)
)

# Drug properties — Chloroquine (reference lysosomotropic agent)
CHLOROQUINE = dict(
    name         = 'Chloroquine',
    MW           = 319.87,
    logP         = 4.63,    # lipophilic
    pKa1         = 10.10,   # stronger basic
    pKa2         = 8.38,    # weaker basic
    fu_plasma    = 0.39,
    B2P          = 2.4,
    dose_oral    = 500.0,   # mg
    F_oral       = 0.89,
    ka           = 0.5,     # h-1
    # True PK (reflects massive tissue accumulation)
    Vc_kg        = 0.5,     # L/kg (misleadingly small — plasma only)
    Vd_ss        = 500.0,   # L/kg (massive — lysosomes!)
    CL_total     = 0.45,    # L/h/kg
)
BW = 70.0

# ── Henderson-Hasselbalch lysosomal partition coefficient ─────────────
def kp_lysosome_hh(pKa, pH_lysosome=4.5, pH_cytosol=7.2):
    """
    Lysosome:cytosol partition coefficient for a monoprotic base.
    HH equation: K = (1 + 10^(pKa - pH_lys)) / (1 + 10^(pKa - pH_cyt))
    Numerator: total drug in lysosome (neutral + ionized)
    Denominator: total drug in cytosol (neutral + ionized)
    """
    K_lys = (1 + 10**(pKa - pH_lysosome)) / (1 + 10**(pKa - pH_cytosol))
    return K_lys

def kp_cytosol_plasma_hh(pKa, logP, pH_plasma=7.4, pH_cytosol=7.2):
    """
    Cytosol:plasma partition coefficient (Rodgers & Rowland).
    Simplified: accounts for pH difference and lipophilicity.
    """
    fu_neutral = 1 / (1 + 10**(pKa - pH_plasma))
    P = 10**logP
    Kp_cyt = (1 + 10**(pKa - pH_cytosol)) / (1 + 10**(pKa - pH_plasma)) * \
              (1 + 0.3 * P) / (1 + 0.3 * P * fu_neutral)
    return max(Kp_cyt, 0.01)

# Calculate partition coefficients for chloroquine
Kp_lys_cyt = kp_lysosome_hh(CHLOROQUINE['pKa1'], PH['lysosome'], PH['cytosol'])
Kp_cyt_pla = kp_cytosol_plasma_hh(CHLOROQUINE['pKa1'], CHLOROQUINE['logP'])
Kp_lys_pla = Kp_lys_cyt * Kp_cyt_pla  # total lysosome:plasma

print('Chloroquine Lysosomal Trapping Parameters:')
print('  pKa1:                   ', CHLOROQUINE['pKa1'])
print('  pH lysosome:            ', PH['lysosome'])
print('  Kp (lysosome/cytosol):  ', round(Kp_lys_cyt, 0))
print('  Kp (cytosol/plasma):    ', round(Kp_cyt_pla, 2))
print('  Kp (lysosome/plasma):   ', round(Kp_lys_pla, 0))
print()
print('  → 1 mg/L plasma → ', round(Kp_lys_pla, 0), 'mg/L lysosome!')

# pKa sensitivity
pka_range = np.arange(5, 12, 0.5)
Kp_lys_range = [kp_lysosome_hh(pka) for pka in pka_range]
print()
print('Lysosomal trapping vs pKa (pH_lys=4.5, pH_cyt=7.2):')
for pka, kp in zip(pka_range, Kp_lys_range):
    stars = '*' * min(int(np.log10(max(kp,1))), 8)
    print(f'  pKa={pka:.1f}  Kp={round(kp):<10} {stars}')

## 2. Extended PBPK Model — Lysosomal Sub-Compartments

Each tissue compartment is split into:
1. **Vascular** (blood-side)
2. **Cytosol** (intracellular, pH 7.2)
3. **Lysosome** (intracellular organelle, pH 4.5)

Drug moves: plasma → cytosol (passive, neutral form) → lysosome (ion trap)

In [ ]:
# Tissue volumes and flows (70 kg human)
TISSUES = dict(
    liver   = dict(V=1.80, Q=1.35*5, lyso_frac=0.015),  # 1.5% lysosome
    muscle  = dict(V=28.5, Q=0.75*5, lyso_frac=0.010),
    fat     = dict(V=10.0, Q=0.25*5, lyso_frac=0.005),
    lung    = dict(V=0.60, Q=5.00,   lyso_frac=0.020),  # high lysosomes
    kidney  = dict(V=0.33, Q=1.10*5, lyso_frac=0.015),
)
CO = 5.0  # L/h cardiac output

# Lysosomal volume fractions (fraction of total tissue volume)
# Literature: lysosomes ~1-2% of cell volume
for tname, t in TISSUES.items():
    t['V_lys']  = t['V'] * t['lyso_frac']   # lysosomal volume
    t['V_cyt']  = t['V'] * (1 - t['lyso_frac'])  # cytosolic volume
    t['PS_lys'] = 0.05 * t['V_lys']          # lysosomal membrane permeability

def build_lyso_pbpk_params(drug, tissues, BW):
    """Build full parameter dict for lysosomal PBPK model."""
    Kp_lc = kp_lysosome_hh(drug['pKa1'], PH['lysosome'], PH['cytosol'])
    Kp_cp = kp_cytosol_plasma_hh(drug['pKa1'], drug['logP'])
    return dict(
        Vc       = drug['Vc_kg'] * BW,
        CL_total = drug['CL_total'] * BW,
        ka       = drug['ka'],
        F_oral   = drug['F_oral'],
        dose     = drug['dose_oral'],
        Kp_lc    = Kp_lc,    # lysosome:cytosol
        Kp_cp    = Kp_cp,    # cytosol:plasma
        Kp_lp    = Kp_lc * Kp_cp,  # lysosome:plasma
        tissues  = tissues,
        BW       = BW,
    )

p_lys = build_lyso_pbpk_params(CHLOROQUINE, TISSUES, BW)

print('Lysosomal PBPK parameters:')
print('  Kp lysosome:cytosol:  ', round(p_lys['Kp_lc'],0))
print('  Kp cytosol:plasma:    ', round(p_lys['Kp_cp'],2))
print('  Kp lysosome:plasma:   ', round(p_lys['Kp_lp'],0))
print()
print('Tissue lysosomal volumes:')
for tname, t in TISSUES.items():
    print(f'  {tname.ljust(8)}: V_lys={t["V_lys"]:.4f} L  '
          f'({t["lyso_frac"]*100:.1f}% of tissue)')

## 3. PBPK ODE System with Lysosomal Compartments

In [ ]:
def lyso_pbpk_odes(y, t, p):
    """
    Extended PBPK with lysosomal sub-compartments.

    State variables:
    y[0]  = Agut       oral absorption depot
    y[1]  = Ac         plasma/central (nmol)
    y[2]  = A_cyt_liv  cytosol liver
    y[3]  = A_lys_liv  lysosome liver
    y[4]  = A_cyt_mus  cytosol muscle
    y[5]  = A_lys_mus  lysosome muscle
    y[6]  = A_cyt_fat  cytosol fat
    y[7]  = A_lys_fat  lysosome fat
    y[8]  = A_cyt_kid  cytosol kidney
    y[9]  = A_lys_kid  lysosome kidney
    """
    (Agut, Ac,
     A_cyt_liv, A_lys_liv,
     A_cyt_mus, A_lys_mus,
     A_cyt_fat, A_lys_fat,
     A_cyt_kid, A_lys_kid) = y

    ts   = p['tissues']
    Vc   = p['Vc']
    Kp_lc= p['Kp_lc']
    Kp_cp= p['Kp_cp']

    # Plasma concentration
    Cp   = max(Ac / Vc, 0)

    # Cytosol concentrations
    Cc_liv= max(A_cyt_liv / ts['liver']['V_cyt'],  0)
    Cc_mus= max(A_cyt_mus / ts['muscle']['V_cyt'], 0)
    Cc_fat= max(A_cyt_fat / ts['fat']['V_cyt'],    0)
    Cc_kid= max(A_cyt_kid / ts['kidney']['V_cyt'], 0)

    # Lysosome concentrations
    Cl_liv= max(A_lys_liv / ts['liver']['V_lys'],  0)
    Cl_mus= max(A_lys_mus / ts['muscle']['V_lys'], 0)
    Cl_fat= max(A_lys_fat / ts['fat']['V_lys'],    0)
    Cl_kid= max(A_lys_kid / ts['kidney']['V_lys'], 0)

    # Gut absorption
    absorb = p['ka'] * Agut * p['F_oral']

    # Plasma↔Cytosol transport (passive, driven by neutral form)
    # Net flux = Q*(Cp - Cc/Kp_cp) [perfusion-limited]
    J_liv = ts['liver']['Q']  * (Cp - Cc_liv / Kp_cp)
    J_mus = ts['muscle']['Q'] * (Cp - Cc_mus / Kp_cp)
    J_fat = ts['fat']['Q']    * (Cp - Cc_fat / Kp_cp)
    J_kid = ts['kidney']['Q'] * (Cp - Cc_kid / Kp_cp)

    # Cytosol↔Lysosome transport (PS-limited, neutral form only)
    # Net flux = PS * (Cc - Cl/Kp_lc)
    L_liv = ts['liver']['PS_lys']  * (Cc_liv - Cl_liv / Kp_lc)
    L_mus = ts['muscle']['PS_lys'] * (Cc_mus - Cl_mus / Kp_lc)
    L_fat = ts['fat']['PS_lys']    * (Cc_fat - Cl_fat / Kp_lc)
    L_kid = ts['kidney']['PS_lys'] * (Cc_kid - Cl_kid / Kp_lc)

    # Elimination (hepatic + renal, from plasma)
    elim  = p['CL_total'] * Cp

    # ODEs
    dAgut     = -p['ka'] * Agut
    dAc       = absorb - J_liv - J_mus - J_fat - J_kid - elim
    dA_cyt_liv= J_liv - L_liv
    dA_lys_liv= L_liv
    dA_cyt_mus= J_mus - L_mus
    dA_lys_mus= L_mus
    dA_cyt_fat= J_fat - L_fat
    dA_lys_fat= L_fat
    dA_cyt_kid= J_kid - L_kid
    dA_lys_kid= L_kid

    return [dAgut, dAc,
            dA_cyt_liv, dA_lys_liv,
            dA_cyt_mus, dA_lys_mus,
            dA_cyt_fat, dA_lys_fat,
            dA_cyt_kid, dA_lys_kid]


t_sim = np.linspace(0, 720, 5000)  # 30 days
t_days= t_sim / 24

DOSE  = CHLOROQUINE['dose_oral']   # 500 mg
y0    = [DOSE] + [0]*9

sol   = odeint(lyso_pbpk_odes, y0, t_sim, args=(p_lys,),
               rtol=1e-6, atol=1e-8, mxstep=8000)

# Extract concentrations (mg/L)
Cp        = np.maximum(sol[:,1] / p_lys['Vc'], 0)
Cc_liv    = np.maximum(sol[:,2] / TISSUES['liver']['V_cyt'],  0)
Cl_liv    = np.maximum(sol[:,3] / TISSUES['liver']['V_lys'],  0)
Cc_mus    = np.maximum(sol[:,4] / TISSUES['muscle']['V_cyt'], 0)
Cl_mus    = np.maximum(sol[:,5] / TISSUES['muscle']['V_lys'], 0)

# Total tissue concentrations (cytosol + lysosome weighted)
def total_tissue_conc(A_cyt, A_lys, V_cyt, V_lys):
    return np.maximum((A_cyt + A_lys) / (V_cyt + V_lys), 0)

C_total_liv = total_tissue_conc(sol[:,2], sol[:,3],
                                 TISSUES['liver']['V_cyt'], TISSUES['liver']['V_lys'])
C_total_mus = total_tissue_conc(sol[:,4], sol[:,5],
                                 TISSUES['muscle']['V_cyt'], TISSUES['muscle']['V_lys'])

AUC_plasma = np.trapezoid(Cp, t_sim)
AUC_lys_liv= np.trapezoid(Cl_liv, t_sim)

print('Lysosomal PBPK simulation (500mg oral, 30 days):')
print('  Peak plasma conc:         ', round(Cp.max(),4), 'mg/L')
print('  Peak lysosome (liver):    ', round(Cl_liv.max(),2), 'mg/L')
print('  Peak lysosome (muscle):   ', round(Cl_mus.max(),2), 'mg/L')
print('  Lysosome/plasma ratio:    ', round(Cl_liv.max()/max(Cp.max(),1e-6),0))
print('  AUC plasma:               ', round(AUC_plasma,2), 'mg*h/L')
print('  AUC lysosome/plasma ratio:', round(AUC_lys_liv/max(AUC_plasma,1e-6),0))

## 4. Model Comparison — With vs Without Lysosomal Trapping

In [ ]:
def simple_pbpk_odes(y, t, p):
    """Standard PBPK without lysosomal compartments (Kp_lc=1)."""
    (Agut, Ac,
     A_cyt_liv, A_lys_liv,
     A_cyt_mus, A_lys_mus,
     A_cyt_fat, A_lys_fat,
     A_cyt_kid, A_lys_kid) = y

    ts   = p['tissues']
    Vc   = p['Vc']
    Kp_cp= p['Kp_cp']

    Cp    = max(Ac / Vc, 0)
    Cc_liv= max(A_cyt_liv / ts['liver']['V_cyt'],  0)
    Cc_mus= max(A_cyt_mus / ts['muscle']['V_cyt'], 0)
    Cc_fat= max(A_cyt_fat / ts['fat']['V_cyt'],    0)
    Cc_kid= max(A_cyt_kid / ts['kidney']['V_cyt'], 0)

    absorb= p['ka'] * Agut * p['F_oral']
    J_liv = ts['liver']['Q']  * (Cp - Cc_liv / Kp_cp)
    J_mus = ts['muscle']['Q'] * (Cp - Cc_mus / Kp_cp)
    J_fat = ts['fat']['Q']    * (Cp - Cc_fat / Kp_cp)
    J_kid = ts['kidney']['Q'] * (Cp - Cc_kid / Kp_cp)
    # No lysosomal flux
    elim  = p['CL_total'] * Cp

    return [-(p['ka']*Agut),
            absorb - J_liv - J_mus - J_fat - J_kid - elim,
            J_liv, 0, J_mus, 0, J_fat, 0, J_kid, 0]

sol_simple = odeint(simple_pbpk_odes, y0, t_sim, args=(p_lys,),
                    rtol=1e-6, atol=1e-8, mxstep=5000)

Cp_simple  = np.maximum(sol_simple[:,1] / p_lys['Vc'], 0)
Cl_liv_simple = np.zeros(len(t_sim))  # no lysosomal compartment

print('Comparison: With vs Without Lysosomal Trapping:')
print('\n  Plasma:')
print('    With lysosomal trapping: Cmax =', round(Cp.max(),4), 'mg/L')
print('    Without lysosomal trap:  Cmax =', round(Cp_simple.max(),4), 'mg/L')
print('    Ratio:', round(Cp.max()/max(Cp_simple.max(),1e-6),3))
print('\n  Liver lysosome peak conc (with trapping):', round(Cl_liv.max(),2), 'mg/L')
print('  Liver cytosol peak conc:', round(Cc_liv.max(),4), 'mg/L')
print('  Lysosome enrichment vs cytosol:', round(Cl_liv.max()/max(Cc_liv.max(),1e-6),0), 'x')

## 5. pKa Sensitivity — Which Drugs Are Lysosomotropic?

In [ ]:
# Known lysosomotropic drugs for context
DRUGS_COMPARISON = {
    'Chloroquine':  dict(pKa=10.1, logP=4.63, category='Antimalarial'),
    'Hydroxychloroquine': dict(pKa=9.67, logP=3.55, category='Antimalarial'),
    'Amiodarone':   dict(pKa=9.0,  logP=7.57, category='Antiarrhythmic'),
    'Amitriptyline':dict(pKa=9.4,  logP=4.92, category='Antidepressant'),
    'Imipramine':   dict(pKa=9.5,  logP=4.28, category='Antidepressant'),
    'Propranolol':  dict(pKa=9.5,  logP=3.48, category='Beta-blocker'),
    'Aspirin':      dict(pKa=3.5,  logP=1.19, category='NSAID (acid, no trap)'),
    'Metformin':    dict(pKa=11.5, logP=-1.43,category='Biguanide (hydrophilic)'),
}

trap_df = pd.DataFrame([
    {
        'Drug': name,
        'pKa': props['pKa'],
        'logP': props['logP'],
        'Category': props['category'],
        'Kp_lys_cyt': round(kp_lysosome_hh(props['pKa']), 0),
        'Lysosomotropic': 'YES' if (props['pKa'] > 6 and props['logP'] > 1) else 'NO'
    }
    for name, props in DRUGS_COMPARISON.items()
])
trap_df = trap_df.sort_values('Kp_lys_cyt', ascending=False)
print('Lysosomotropic potential by drug:')
print(trap_df.to_string(index=False))

## 6. Visualization

In [ ]:
BLUE='#2563EB'; RED='#DC2626'; GREEN='#16A34A'
AMBER='#D97706'; PURP='#7C3AED'; TEAL='#0D9488'

fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(3, 3, hspace=0.45, wspace=0.38)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, 0])
ax3 = fig.add_subplot(gs[1, 1])
ax4 = fig.add_subplot(gs[1, 2])
ax5 = fig.add_subplot(gs[2, 0])
ax6 = fig.add_subplot(gs[2, 1])
ax7 = fig.add_subplot(gs[2, 2])

# Panel 1: Multi-compartment concentrations over 30 days
ax1.semilogy(t_days, Cp+1e-6,       color=BLUE,  lw=2.5, label='Plasma')
ax1.semilogy(t_days, Cc_liv+1e-6,   color=GREEN, lw=2,   label='Liver cytosol')
ax1.semilogy(t_days, Cl_liv+1e-6,   color=RED,   lw=2.5, label='Liver LYSOSOME')
ax1.semilogy(t_days, Cc_mus+1e-6,   color=AMBER, lw=2,   label='Muscle cytosol')
ax1.semilogy(t_days, Cl_mus+1e-6,   color=PURP,  lw=2,   ls='--', label='Muscle lysosome')
ax1.set(xlabel='Time (days)', ylabel='Concentration (mg/L) [log]',
        title='Chloroquine — Lysosomal Trapping PBPK (500mg oral, 30 days)\n'
              'Plasma · Cytosol · Lysosome across tissues')
ax1.title.set_fontweight('bold')
ax1.legend(fontsize=9, ncol=3); ax1.grid(True, alpha=0.25, which='both')

# Panel 2: pKa vs Kp_lysosome
pka_fine = np.linspace(4, 12, 200)
kp_fine  = [kp_lysosome_hh(p) for p in pka_fine]
ax2.semilogy(pka_fine, kp_fine, color=RED, lw=2.5)
ax2.axvline(6, color='gray', ls='--', lw=1.5, label='pKa=6 threshold')
for dname, props in list(DRUGS_COMPARISON.items())[:6]:
    kp = kp_lysosome_hh(props['pKa'])
    ax2.scatter(props['pKa'], kp, s=80, zorder=5)
    ax2.annotate(dname, (props['pKa'], kp),
                 textcoords='offset points', xytext=(4,2), fontsize=7)
ax2.set(xlabel='pKa (basic drug)', ylabel='Kp lysosome/cytosol [log]',
        title='Henderson-Hasselbalch\npKa vs Lysosomal Kp')
ax2.title.set_fontweight('bold')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.25, which='both')

# Panel 3: pH effect on Kp
ph_range = np.linspace(4.0, 6.5, 100)
kp_ph    = [kp_lysosome_hh(10.1, pH_lysosome=ph) for ph in ph_range]
ax3.semilogy(ph_range, kp_ph, color=PURP, lw=2.5)
ax3.axvline(4.5, color='red', ls='--', lw=1.5, label='Normal pH 4.5')
ax3.axvline(5.5, color='orange', ls='--', lw=1.5, label='Disturbed pH 5.5')
ax3.set(xlabel='Lysosomal pH', ylabel='Kp lysosome/cytosol [log]',
        title='Lysosomal pH Effect on Kp\n(Chloroquine, pKa=10.1)')
ax3.title.set_fontweight('bold')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.25, which='both')

# Panel 4: With vs without lysosomal trapping (plasma)
ax4.semilogy(t_days, Cp+1e-6,        color=RED,  lw=2.5, label='With lysosomal trapping')
ax4.semilogy(t_days, Cp_simple+1e-6, color=BLUE, lw=2.5, ls='--', label='Without trapping')
ax4.set(xlabel='Time (days)', ylabel='Plasma conc (mg/L) [log]',
        title='Plasma PK\nWith vs Without Lysosomal Trapping')
ax4.title.set_fontweight('bold')
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.25, which='both')

# Panel 5: Tissue distribution at peak (bar chart)
t_peak_idx = np.argmax(Cp)
tissue_concs = {
    'Plasma':          Cp[t_peak_idx],
    'Liver\ncytosol':  Cc_liv[t_peak_idx],
    'Liver\nlysosome': Cl_liv[t_peak_idx],
    'Muscle\ncytosol': Cc_mus[t_peak_idx],
    'Muscle\nlysosome':Cl_mus[t_peak_idx],
}
bar_colors = [BLUE, GREEN, RED, AMBER, PURP]
ax5.bar(tissue_concs.keys(), tissue_concs.values(),
        color=bar_colors, alpha=0.85)
ax5.set_yscale('log')
ax5.set(ylabel='Concentration (mg/L) [log]',
        title='Tissue Distribution at Plasma Cmax\n(Massive lysosomal accumulation)')
ax5.title.set_fontweight('bold'); ax5.grid(True, alpha=0.25, axis='y')

# Panel 6: Lysosomotropic drug comparison (bar chart)
trap_sorted = trap_df[trap_df['Lysosomotropic']=='YES'].head(6)
ax6.barh(trap_sorted['Drug'], np.log10(trap_sorted['Kp_lys_cyt'].clip(1)),
         color=RED, alpha=0.8)
ax6.set(xlabel='log10(Kp lysosome/cytosol)',
        title='Lysosomotropic Potential\n(Selected drugs, pH_lys=4.5)')
ax6.title.set_fontweight('bold'); ax6.grid(True, alpha=0.25, axis='x')

# Panel 7: Compartment diagram
ax7.axis('off')
diagram = (
    'LYSOSOMAL TRAPPING MODEL\n\n'
    'Plasma (pH 7.4)\n'
    '  C = A/Vc\n'
    '  ↕ perfusion-limited\n\n'
    'Cytosol (pH 7.2)\n'
    '  Kp_cp = ' + str(round(p_lys['Kp_cp'],2)) + '\n'
    '  ↕ PS-limited (neutral form)\n\n'
    'Lysosome (pH 4.5)\n'
    '  Kp_lys/cyt = ' + str(int(p_lys['Kp_lc'])) + '\n'
    '  ION TRAP: drug-H+ cannot exit\n\n'
    'HH equation:\n'
    '  Kp = (1+10^(pKa-pH_lys))\n'
    '       / (1+10^(pKa-pH_cyt))\n\n'
    'Chloroquine Kp_lys/plasma:\n'
    '  ~' + str(int(p_lys['Kp_lp'])) + 'x enrichment'
)
ax7.text(0.05, 0.95, diagram, transform=ax7.transAxes,
         fontsize=9.5, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax7.set_title('Model Structure & Parameters', fontweight='bold')

plt.suptitle(
    'Lysosomal Trapping PBPK — Chloroquine Intracellular Distribution\n'
    'Henderson-Hasselbalch pH Partitioning · Extended PBPK | OSP MoBi Exercise',
    fontsize=13, fontweight='bold', y=1.01
)
plt.savefig('lysosomal_trapping_pbpk.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: lysosomal_trapping_pbpk.png')

## 7. Interactive Dashboard

In [ ]:
fig_p = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Lysosomal PBPK — All Compartments',
        'pKa vs Lysosomal Kp (HH equation)',
        'Plasma PK: With vs Without Trapping',
        'Lysosomotropic Drug Comparison'
    ),
    vertical_spacing=0.18, horizontal_spacing=0.12
)

for arr, name, color, dash in [
    (Cp,      'Plasma',          BLUE,  'solid'),
    (Cc_liv,  'Liver cytosol',   GREEN, 'solid'),
    (Cl_liv,  'Liver lysosome',  RED,   'solid'),
    (Cc_mus,  'Muscle cytosol',  AMBER, 'solid'),
    (Cl_mus,  'Muscle lysosome', PURP,  'dash'),
]:
    fig_p.add_trace(go.Scatter(
        x=t_days, y=arr+1e-6, mode='lines', name=name,
        line=dict(color=color, width=2, dash=dash),
        hovertemplate=name+'<br>%{x:.1f}d: %{y:.5f} mg/L<extra></extra>'
    ), row=1, col=1)

fig_p.add_trace(go.Scatter(
    x=pka_fine, y=kp_fine, mode='lines', name='HH curve',
    line=dict(color=RED, width=2)
), row=1, col=2)
for dname, props in list(DRUGS_COMPARISON.items())[:6]:
    kp = kp_lysosome_hh(props['pKa'])
    fig_p.add_trace(go.Scatter(
        x=[props['pKa']], y=[kp], mode='markers+text',
        text=[dname], textposition='top right',
        marker=dict(size=10), showlegend=False
    ), row=1, col=2)

fig_p.add_trace(go.Scatter(
    x=t_days, y=Cp+1e-6, mode='lines', name='With trapping',
    line=dict(color=RED, width=2)
), row=2, col=1)
fig_p.add_trace(go.Scatter(
    x=t_days, y=Cp_simple+1e-6, mode='lines', name='Without trapping',
    line=dict(color=BLUE, width=2, dash='dash')
), row=2, col=1)

fig_p.add_trace(go.Bar(
    x=np.log10(trap_sorted['Kp_lys_cyt'].values.clip(1)),
    y=trap_sorted['Drug'].values,
    orientation='h', marker_color=RED, opacity=0.8,
    showlegend=False,
    hovertemplate='%{y}: log10(Kp)=%{x:.1f}<extra></extra>'
), row=2, col=2)

for ri,ci,xl,yl in [
    (1,1,'Time (days)','Conc (mg/L)'),
    (1,2,'pKa','Kp lys/cyt'),
    (2,1,'Time (days)','Plasma (mg/L)'),
    (2,2,'log10(Kp lys/cyt)','Drug')
]:
    fig_p.update_xaxes(title_text=xl, row=ri, col=ci)
    fig_p.update_yaxes(title_text=yl, row=ri, col=ci)
for ri,ci in [(1,1),(1,2),(2,1)]:
    fig_p.update_yaxes(type='log', row=ri, col=ci)

fig_p.update_layout(
    title=dict(
        text='Lysosomal Trapping PBPK -- Interactive Dashboard<br>'
             '<sup>Chloroquine | HH partitioning | Intracellular accumulation | OSP MoBi Exercise</sup>',
        font=dict(size=14)
    ),
    height=720, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, x=0)
)
fig_p.show()
fig_p.write_html('lysosomal_trapping_dashboard.html')
print('Saved: lysosomal_trapping_dashboard.html')

## 8. Export

In [ ]:
df_out = pd.DataFrame({
    'time_days':      t_days,
    'C_plasma':       Cp,
    'C_liver_cytosol':Cc_liv,
    'C_liver_lysosome':Cl_liv,
    'C_muscle_cytosol':Cc_mus,
    'C_muscle_lysosome':Cl_mus,
})
df_out.to_csv('lysosomal_trapping_pk.csv', index=False)
trap_df.to_csv('lysosomotropic_drugs.csv', index=False)

print('Key Parameters:')
print('  Kp lysosome/cytosol (HH):', int(p_lys['Kp_lc']))
print('  Kp lysosome/plasma:      ', int(p_lys['Kp_lp']))
print('  Peak lysosome/plasma ratio:', round(Cl_liv.max()/max(Cp.max(),1e-6),0))
print()
print('Lysosomotropic drug ranking:')
print(trap_df[['Drug','pKa','logP','Kp_lys_cyt','Lysosomotropic']].to_string(index=False))

## Key Findings

| Compartment | Concentration vs plasma | Mechanism |
|---|---|---|
| Cytosol | ~2-5x | pH 7.2 vs 7.4 differential |
| Lysosome | ~100,000x | Ion trapping pH 4.5 |
| Total tissue | ~1000x | Dominated by lysosomal fraction |

## MoBi Parallel Steps
1. Build standard 2-compartment PBPK model
2. Add lysosomal sub-container to each tissue organ
3. Set lysosomal volume (1-2% of tissue volume)
4. Add pH parameters: pH_lysosome=4.5, pH_cytosol=7.2, pKa=drug
5. Define Kp_lysosome formula: (1+10^(pKa-pH_lys))/(1+10^(pKa-pH_cyt))
6. Add transport from cytosol to lysosome (PS-limited, neutral form)
7. Simulate — observe massive lysosomal accumulation
8. Compare plasma PK with vs without lysosomal model

## References
1. OSP MoBi Course: Lysosomal Trapping (v12)
2. de Duve C et al. Basic drugs and lysosomes. Biochem Pharmacol 1974
3. Kaufmann AM, Krise JP. Lysosomal sequestration. J Pharm Sci 2007
4. Trapp S et al. Prediction of lysosomal volume of distribution. AAPS J 2008
5. Daniel WA. Lysosomal trapping and drug accumulation. Pharmacol Rep 2003

---
*Nadia Tasnim Ahmed, PhD · github.com/ahmedn12*